In [41]:
import numpy as np
import pandas as pd

from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
#This transforms data
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler

import time

In [42]:
# dfPandas = pd.read_csv('data/diabetes_012.csv')

# dfPandas.head(5)

In [43]:
builder = SparkSession.builder
builder = builder.config("spark.executor.memory", "2G")
builder = builder.config("spark.driver.memory", "10G")
builder = builder.config("spark.driver.maxResultSize", "5G")
spark = builder.appName("RandomForest").getOrCreate()
sc = spark.sparkContextsc = spark.sparkContext

In [44]:
#Activate Spark
# spark = SparkSession.builder.appName("RandomForest").config("spark.executor.instances", "4").getOrCreate()
# spark = SparkSession.builder.appName("RandomForest").getOrCreate()

# spark = SparkSession.builder.appName("RandomForest").config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.minExecutors", "1").config("spark.dynamicAllocation.maxExecutors", "3").config("spark.executor.memory", "4g").getOrCreate()

# spark.conf.set("spark.dynamicAllocation.enabled", "true")
# spark.conf.set("spark.dynamicAllocation.minExecutors", "1")
# spark.conf.set("spark.dynamicAllocation.maxExecutors", "10")
# # Requires external shuffle service to be enabled in most environments
# spark.conf.set("spark.shuffle.service.enabled", "true")

In [45]:
# spark.stop()

In [46]:
df = spark.read.csv('data/diabetes_012.csv',inferSchema=True, header=True)

In [47]:
# df = df.sample(withReplacement=False, fraction=0.3, seed=42)

In [48]:
# df.show(5)

In [49]:
inputColumns=["HighBP", "HighChol", "CholCheck","BMI","Smoker","Stroke","HeartDiseaseorAttack","PhysActivity","Fruits","Veggies","HvyAlcoholConsump","AnyHealthcare","NoDocbcCost","GenHlth","MentHlth","PhysHlth","DiffWalk","Sex", "Age","Education","Income"]

In [50]:
assembler = VectorAssembler(
    inputCols=inputColumns,
    outputCol="features")
vectorDF = assembler.transform(df)

In [51]:
vectorDF = vectorDF.drop(*inputColumns)
vectorDF.show(5)

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|[1.0,1.0,1.0,40.0...|
|  0.0|(21,[3,4,7,12,13,...|
|  0.0|[1.0,1.0,1.0,28.0...|
|  0.0|(21,[0,2,3,7,8,9,...|
|  0.0|[1.0,1.0,1.0,24.0...|
+-----+--------------------+
only showing top 5 rows


In [52]:
scaler = StandardScaler(
    inputCol="features", 
    outputCol="scaledFeatures",
    withStd=True, 
    withMean=True
)

scaler_model = scaler.fit(vectorDF)

vectorDF = scaler_model.transform(vectorDF).drop("features")
# vectorDF = vectorDF.cache()
# broadcastDF = sc.broadcast(vectorDF)

In [53]:
vectorDF.show(5)

+-----+--------------------+
|label|      scaledFeatures|
+-----+--------------------+
|  0.0|[1.15368587013306...|
|  0.0|[-0.8667836574183...|
|  0.0|[1.15368587013306...|
|  0.0|[1.15368587013306...|
|  0.0|[1.15368587013306...|
+-----+--------------------+
only showing top 5 rows


In [54]:
forest = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label")

In [55]:
paramGrid = (ParamGridBuilder()
             .addGrid(forest.numTrees, [100, 200, 300])
             .addGrid(forest.maxDepth, [5, 10, 15, 20])
             .build())

paramGrid = (ParamGridBuilder()
             .addGrid(forest.numTrees, [100, 200])
             .addGrid(forest.maxDepth, [5, 10, 15])
             .addGrid(forest.maxBins, [32, 40, 50, 64])
             .build())

evaluator = MulticlassClassificationEvaluator(metricName="accuracy")

# cv = CrossValidator(estimator=forest,
#                     estimatorParamMaps=paramGrid,
#                     evaluator=evaluator,
#                     numFolds=3)

cv = TrainValidationSplit(estimator=forest,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    trainRatio=0.8)

In [56]:

# Run grid search
startTime = time.perf_counter()
cvModel = cv.fit(vectorDF)

# Access the best model
bestModel = cvModel.bestModel

endTime = time.perf_counter()
print(f"Elapsed time: {endTime - startTime:.6f} seconds")

26/04/13 07:57:51 WARN DAGScheduler: Broadcasting large task binary with size 1063.4 KiB
26/04/13 07:57:53 WARN DAGScheduler: Broadcasting large task binary with size 2040.9 KiB
26/04/13 07:57:55 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
26/04/13 07:57:57 WARN DAGScheduler: Broadcasting large task binary with size 1172.4 KiB
26/04/13 07:57:57 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
26/04/13 07:58:01 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/13 07:58:01 WARN DAGScheduler: Broadcasting large task binary with size 13.8 MiB
26/04/13 07:58:05 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/04/13 07:58:10 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
26/04/13 07:58:14 WARN DAGScheduler: Broadcasting large task binary with size 1063.5 KiB
26/04/13 07:58:15 WARN DAGScheduler: Broadcasting large task binary with size 2041.0 KiB
26/04/13 07:58:17 WARN DAGScheduler: B

Elapsed time: 1627.095442 seconds


In [57]:
#204 secs vs 498 seconds with
summary = bestModel.summary
# print(summary.objectiveHistory)
# print(f"RMSE: {summary.rootMeanSquaredError}")
# print(f"R2: {summary.r2}")
# summary.residuals.show() # Show residuals
# print(bestModel.params)

params = bestModel.extractParamMap()

# Print parameters without docstrings
for param, value in params.items():
    print(f"{param.name}: {value}",end=", ")
    
print('\naccuracy:', summary.accuracy)
print('precision:',summary.weightedPrecision)
print(summary.totalIterations)

bootstrap: True, cacheNodeIds: False, checkpointInterval: 10, featureSubsetStrategy: auto, featuresCol: scaledFeatures, impurity: gini, labelCol: label, leafCol: , maxBins: 32, maxDepth: 15, maxMemoryInMB: 256, minInfoGain: 0.0, minInstancesPerNode: 1, minWeightFractionPerNode: 0.0, numTrees: 100, predictionCol: prediction, probabilityCol: probability, rawPredictionCol: rawPrediction, seed: -4983835079492653944, subsamplingRate: 1.0, 

26/04/13 08:24:30 WARN DAGScheduler: Broadcasting large task binary with size 85.4 MiB
[Stage 2458:=========>                                              (1 + 5) / 6]


accuracy: 0.8769315673289183
precision: 0.8749098074190984
0


In [58]:
# spark.stop()